In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade boto3 pyarrow
!pip install --proxy=192.168.2.10:8080 --upgrade seaborn statsmodels

In [ ]:
import datetime
from io import BytesIO
import itertools

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

import warnings
warnings.filterwarnings('ignore')

# Configuration

In [ ]:
portfolio = 'Buckeye'
sub_portfolio = 'LP'

first_input_date = '2021-01-31'
last_input_date = '2024-12-31'

TARGET_METRIC = 'Gross Credit Loss Rate'
ACTIVE_ACCOUNT_TYPE = 'Active Accounts'

bucket_name = 'fmsp-sagemaker-rstudio-01'
accounts_file_key = 'Forecasting/Buckeye/Buckeye_combined_data_for_multivariate_forecasting.parquet'
metrics_file_key = 'Forecasting/Buckeye/Forecasting_Buckeye_combined_with_predata_v4.parquet'

print('Configured target:', TARGET_METRIC)

# Load Accounts Dataset (S3)

In [ ]:
s3 = boto3.client('s3')
obj_accounts = s3.get_object(Bucket=bucket_name, Key=accounts_file_key)
accounts_df = pd.read_parquet(BytesIO(obj_accounts['Body'].read()))
accounts_df['DATE'] = pd.to_datetime(accounts_df['DATE']).dt.date

print(f'Accounts rows loaded: {len(accounts_df):,}')
accounts_df.head()

# Load Metric Dataset (S3)

In [ ]:
obj_metrics = s3.get_object(Bucket=bucket_name, Key=metrics_file_key)
metrics_df = pd.read_parquet(BytesIO(obj_metrics['Body'].read()))
metrics_df['DATE'] = pd.to_datetime(metrics_df['DATE']).dt.date

print(f'Metric rows loaded: {len(metrics_df):,}')
metrics_df.head()

# Prepare Active Accounts Series and Target Series

In [ ]:
# Active accounts series
acc_mask = (
    (accounts_df['PORTFOLIO'] == portfolio) &
    (accounts_df['SUB_PORTFOLIO'] == sub_portfolio) &
    (accounts_df['FORECAST_TYPE'] == 'Actual') &
    (accounts_df['ACCOUNT_TYPE'] == ACTIVE_ACCOUNT_TYPE)
)
acc_series = (
    accounts_df.loc[acc_mask, ['DATE', 'NO_OF_ACCOUNTS']]
    .groupby('DATE', as_index=True)['NO_OF_ACCOUNTS']
    .sum()
    .sort_index()
    .astype(float)
)
acc_series.name = 'active_accounts'

# Target series (Gross Credit Loss Rate)
target_mask = (
    (metrics_df['PORTFOLIO'] == portfolio) &
    (metrics_df['SUB_PORTFOLIO'] == sub_portfolio) &
    (metrics_df['FORECAST_TYPE'] == 'Actual') &
    (metrics_df['METRIC'] == TARGET_METRIC)
)
target_series = (
    metrics_df.loc[target_mask, ['DATE', 'METRIC_VALUE']]
    .groupby('DATE', as_index=True)['METRIC_VALUE']
    .sum()
    .sort_index()
    .astype(float)
)
target_series.name = TARGET_METRIC

# Keep common dates and apply date window
combined = pd.concat([target_series, acc_series], axis=1).dropna()
combined = combined[combined.index >= datetime.date.fromisoformat(first_input_date)]

train_df = combined[combined.index <= datetime.date.fromisoformat(last_input_date)].copy()
test_df = combined[combined.index > datetime.date.fromisoformat(last_input_date)].copy()

print(f'Train rows: {len(train_df)}  ({train_df.index.min()} -> {train_df.index.max()})')
print(f'Test rows : {len(test_df)}  ({test_df.index.min()} -> {test_df.index.max()})')
display(train_df.head())

# Helpers

In [ ]:
def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100)


def print_metrics(label: str, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    print(f'  {label}')
    print(f'    MAPE  : {mean_absolute_percentage_error(y_true, y_pred) * 100:.2f}%')
    print(f'    SMAPE : {smape(y_true, y_pred):.2f}%')
    print(f'    MAE   : {mean_absolute_error(y_true, y_pred):>14,.6f}')
    print(f'    RMSE  : {np.sqrt(mean_squared_error(y_true, y_pred)):>14,.6f}')


def get_benchmark(test_dates):
    mask = (
        (metrics_df['PORTFOLIO'] == portfolio) &
        (metrics_df['SUB_PORTFOLIO'] == sub_portfolio) &
        (metrics_df['METRIC'] == TARGET_METRIC) &
        (metrics_df['FORECAST_TYPE'] == '2025 0+12')
    )
    bdf = metrics_df.loc[mask, ['DATE', 'METRIC_VALUE']].copy()
    bdf['FORECAST_TYPE'] = '2025 0+12'
    return bdf[bdf['DATE'].isin(test_dates)]


def build_pred_df(forecast_values, test_series, pred_label='Prediction'):
    actual_rows = pd.DataFrame({
        'DATE': test_series.index,
        'FORECAST_TYPE': 'Actual',
        'METRIC_VALUE': test_series.values,
    })
    pred_rows = pd.DataFrame({
        'DATE': test_series.index,
        'FORECAST_TYPE': pred_label,
        'METRIC_VALUE': np.asarray(forecast_values, dtype=float),
    })
    bench_rows = get_benchmark(test_series.index)
    out = pd.concat([actual_rows, pred_rows, bench_rows], ignore_index=True)
    return out


def plot_results(pred_df: pd.DataFrame, title: str) -> None:
    palette = {'Actual': '#1f77b4', '2025 0+12': '#2ca02c'}
    default_color = '#ff7f0e'

    fig, ax = plt.subplots(figsize=(18, 6))
    ax.set_title(title, fontsize=14, fontweight='bold')

    labels = pred_df['FORECAST_TYPE'].unique().tolist()
    if 'Actual' in labels:
        labels = ['Actual'] + [x for x in labels if x != 'Actual']

    for label in labels:
        subset = pred_df[pred_df['FORECAST_TYPE'] == label].sort_values('DATE')
        if subset.empty:
            continue
        color = palette.get(label, default_color)
        ax.plot(subset['DATE'].astype(str), subset['METRIC_VALUE'].values, marker='o', linewidth=2, label=label, color=color)

    ax.set_xlabel('DATE')
    ax.set_ylabel('METRIC_VALUE')
    ax.tick_params(axis='x', rotation=90)
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.4f}'))
    plt.tight_layout()
    plt.show()


def evaluate_pred_df(pred_df: pd.DataFrame, model_label: str) -> None:
    y_actual = (
        pred_df[pred_df['FORECAST_TYPE'] == 'Actual']
        .sort_values('DATE')
        .set_index('DATE')['METRIC_VALUE']
    )

    print('\n' + '=' * 60)
    print(f'Error Metrics - {TARGET_METRIC}')
    print('=' * 60)

    for lbl in [model_label, '2025 0+12']:
        subset = pred_df[pred_df['FORECAST_TYPE'] == lbl]
        if subset.empty:
            print(f'\n  {lbl} - no data')
            continue
        y_pred = subset.sort_values('DATE').set_index('DATE')['METRIC_VALUE']
        common = y_actual.index.intersection(y_pred.index)
        if len(common) == 0:
            print(f'\n  {lbl} - no overlapping dates')
            continue
        print_metrics(lbl, y_actual.loc[common].values.astype(float), y_pred.loc[common].values.astype(float))

    print('=' * 60)

# Step 1: Forecast Active Accounts (Exogenous Variable)

This model is the helper model for exogenous input forecasting.
Use your already tuned parameters here.

In [ ]:
acc_train_y = train_df['active_accounts'].astype(float)
acc_test_y = test_df['active_accounts'].astype(float)

# Optional step dummy, similar to your screenshot pattern
step_date = datetime.date.fromisoformat('2023-07-31')
acc_train_exog = pd.DataFrame({'Portfolio_Acquired': (acc_train_y.index <= step_date).astype(int)}, index=acc_train_y.index)
acc_test_exog = pd.DataFrame({'Portfolio_Acquired': (acc_test_y.index <= step_date).astype(int)}, index=acc_test_y.index)

# Use tuned account-model params
ACC_ORDER = (1, 1, 1)
ACC_SEASONAL_ORDER = (1, 1, 0, 12)

acc_model = SARIMAX(
    endog=acc_train_y,
    exog=acc_train_exog,
    order=ACC_ORDER,
    seasonal_order=ACC_SEASONAL_ORDER,
    trend='n',
    enforce_stationarity=True,
    enforce_invertibility=True,
)
acc_result = acc_model.fit(disp=False)
print(acc_result.summary())

acc_fc = acc_result.get_forecast(steps=len(acc_test_y), exog=acc_test_exog).predicted_mean
acc_fc.index = acc_test_y.index

acc_compare = pd.DataFrame({
    'actual_active_accounts': acc_test_y,
    'forecast_active_accounts': acc_fc,
})
display(acc_compare.head())

# Step 2: Main SARIMAX for Gross Credit Loss Rate

No lag_1, no ma_3, no month_num.
Only exogenous input is Active Accounts.

Train exog uses actual active accounts.
Test exog uses forecasted active accounts from Step 1.

In [ ]:
main_train_y = train_df[TARGET_METRIC].astype(float)
main_test_y = test_df[TARGET_METRIC].astype(float)

main_exog_train = pd.DataFrame({'active_accounts': acc_train_y.values}, index=acc_train_y.index)
main_exog_test = pd.DataFrame({'active_accounts': acc_fc.values}, index=acc_fc.index)

MAIN_ORDER = (1, 1, 1)
MAIN_SEASONAL_ORDER = (1, 1, 0, 12)

main_model = SARIMAX(
    endog=main_train_y,
    exog=main_exog_train,
    order=MAIN_ORDER,
    seasonal_order=MAIN_SEASONAL_ORDER,
    trend='n',
    enforce_stationarity=False,
    enforce_invertibility=False,
)
main_result = main_model.fit(disp=False)
print(main_result.summary())

main_fc = main_result.get_forecast(steps=len(main_test_y), exog=main_exog_test).predicted_mean
main_fc.index = main_test_y.index

pred_df_main = build_pred_df(main_fc.values, main_test_y, pred_label='SARIMAX_active_accounts')
pred_df_main

In [ ]:
plot_results(pred_df_main, f'SARIMAX + Active Accounts exog - {TARGET_METRIC}')
evaluate_pred_df(pred_df_main, 'SARIMAX_active_accounts')

# Hyperparameter Tuning (Main Model)

Goal: keep reproducible grid search, optimize MAPE and pattern capture together.

Pattern capture proxy uses:
- variability match (std ratio)
- month-to-month movement match (first-difference correlation)

Ranking score:
composite_score = mape + w_amp * amp_penalty + w_shape * shape_penalty

Lower composite score is better.

In [ ]:
# Reproducible, simple grid
grid = list(itertools.product(
    [0, 1, 2],  # p
    [0, 1],     # d
    [0, 1, 2],  # q
    [0, 1],     # P
    [0, 1],     # D
    [0, 1, 2],  # Q
))

# Weights to balance MAPE and fluctuation capture
w_amp = 20.0
w_shape = 20.0

rows = []
y_true = main_test_y.values.astype(float)

for p, d, q, P, D, Q in grid:
    try:
        model = SARIMAX(
            endog=main_train_y,
            exog=main_exog_train,
            order=(p, d, q),
            seasonal_order=(P, D, Q, 12),
            trend='n',
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False)

        y_pred = model.get_forecast(steps=len(main_test_y), exog=main_exog_test).predicted_mean.values.astype(float)

        mape = float(mean_absolute_percentage_error(y_true, y_pred) * 100.0)

        true_std = float(np.std(y_true)) + 1e-9
        pred_std = float(np.std(y_pred))
        amp_penalty = abs((pred_std / true_std) - 1.0)

        y_true_diff = np.diff(y_true)
        y_pred_diff = np.diff(y_pred)
        if np.std(y_true_diff) < 1e-9 or np.std(y_pred_diff) < 1e-9:
            diff_corr = 0.0
        else:
            diff_corr = float(np.corrcoef(y_true_diff, y_pred_diff)[0, 1])
            if np.isnan(diff_corr):
                diff_corr = 0.0

        shape_penalty = 1.0 - max(diff_corr, 0.0)

        composite_score = mape + (w_amp * amp_penalty) + (w_shape * shape_penalty)

        rows.append({
            'p': p, 'd': d, 'q': q,
            'P': P, 'D': D, 'Q': Q,
            'MAPE': round(mape, 4),
            'amp_penalty': round(float(amp_penalty), 4),
            'shape_penalty': round(float(shape_penalty), 4),
            'score': round(float(composite_score), 4),
            'n_test_points': int(len(main_test_y)),
        })
    except Exception:
        continue

if len(rows) == 0:
    print('No valid parameter combinations were fitted. Reduce grid size and try again.')
    tune_df = pd.DataFrame(columns=['p','d','q','P','D','Q','MAPE','amp_penalty','shape_penalty','score','n_test_points'])
else:
    tune_df = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
    best = tune_df.iloc[0]
    print('Best Params (MAPE + fluctuation capture):')
    print(f"  order=({int(best.p)},{int(best.d)},{int(best.q)})")
    print(f"  seasonal_order=({int(best.P)},{int(best.D)},{int(best.Q)},12)")
    print(f"  MAPE={best.MAPE:.4f}% | amp_penalty={best.amp_penalty:.4f} | shape_penalty={best.shape_penalty:.4f} | score={best.score:.4f}")

print('\nTop 10 by score:')
display(tune_df.head(10))

# Refit Main Model with Tuned Parameters (Optional)

In [ ]:
if len(tune_df) > 0:
    bp = tune_df.iloc[0]
    tuned_order = (int(bp.p), int(bp.d), int(bp.q))
    tuned_seasonal = (int(bp.P), int(bp.D), int(bp.Q), 12)

    tuned_model = SARIMAX(
        endog=main_train_y,
        exog=main_exog_train,
        order=tuned_order,
        seasonal_order=tuned_seasonal,
        trend='n',
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)

    tuned_fc = tuned_model.get_forecast(steps=len(main_test_y), exog=main_exog_test).predicted_mean
    tuned_fc.index = main_test_y.index

    pred_df_tuned = build_pred_df(tuned_fc.values, main_test_y, pred_label='SARIMAX_active_accounts_tuned')
    plot_results(pred_df_tuned, f'Tuned SARIMAX + Active Accounts - {TARGET_METRIC}')
    evaluate_pred_df(pred_df_tuned, 'SARIMAX_active_accounts_tuned')
else:
    print('No tuned parameters available yet.')